In [ ]:
!pip install pandas geopandas libpysal esda scikit-learn matplotlib seaborn mgwr statsmodels hdbscan

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import libpysal as lp
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

# Step 1: Load and Prepare the Dataset
# Load your dataset (replace 'your_data.csv' with the actual file path)
df = pd.read_csv('/content/hdb_resale_latest_transactions.csv')

# Create a GeoDataFrame with geometry from x and y coordinates (SVY21)
df['geometry'] = df.apply(lambda row: Point(row['x'], row['y']), axis=1)
gdf = gpd.GeoDataFrame(df, geometry='geometry')

# Set the coordinate reference system (CRS) to SVY21 (EPSG:3414)
gdf.set_crs(epsg=3414, inplace=True)

# Step 2: Feature Engineering

# A. Distance-Based Features
# Select and standardize distance features
distance_features = ['distance_to_mrt_meters', 'distance_to_cbd', 'distance_to_pri_school_meters']
scaler = StandardScaler()
gdf[distance_features] = scaler.fit_transform(gdf[distance_features])

# B. Spatial Lag Features
# Create spatial weights matrix using k-nearest neighbors (k=5)
w = lp.weights.KNN.from_dataframe(gdf, k=40)

# Compute spatial lag for price_per_sqft (since it's the target)
gdf['price_per_sqft_lag'] = lp.weights.lag_spatial(w, gdf['price_per_sqft'])

# C. Categorical Encoding
# Label encoding for specified categorical variables only
le = LabelEncoder()
gdf['planning_area_ura_encoded'] = le.fit_transform(gdf['planning_area_ura'])
gdf['flat_type_encoded'] = le.fit_transform(gdf['flat_type'])

# Step 3: Define Features and Target
# Select features for the machine learning model (excluding temporal features and resale_price_lag)
features = [
    'distance_to_mrt_meters', 'distance_to_cbd', 'distance_to_pri_school_meters',
    'price_per_sqft_lag',
    'planning_area_ura_encoded', 'flat_type_encoded',
    'floor_area_sqm'
]
target = 'price_per_sqft'  # Target variable set to price_per_sqft

# Step 4: Handle Missing Values
# Drop rows with missing values in selected features or target
gdf = gdf.dropna(subset=features + [target])

# Step 5: Split the Data
# Split into training and testing sets
X = gdf[features]
y = gdf[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Data is now ready for machine learning models!
print("Feature engineering complete. Training and testing sets prepared.")
print(f"Training set size: {X_train.shape[0]} samples")
print(f"Testing set size: {X_test.shape[0]} samples")

Feature engineering complete. Training and testing sets prepared.
Training set size: 55665 samples
Testing set size: 13917 samples


/usr/local/lib/python3.11/dist-packages/libpysal/weights/distance.py:153: UserWarning: The weights matrix is not fully connected: 
 There are 72 disconnected components.
  W.__init__(self, neighbors, id_order=ids, **kwargs)


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from esda.moran import Moran
import libpysal as lp

# Step 1: Fit OLS Regression Model
# Use the training data from feature engineering (X_train, y_train)
ols_model = LinearRegression()
ols_model.fit(X_train, y_train)

# Predict on training and test sets
y_train_pred = ols_model.predict(X_train)
y_test_pred = ols_model.predict(X_test)

# Step 2: Compute Regression Metrics
# Training metrics
r2_train = r2_score(y_train, y_train_pred)
n_train, p = X_train.shape
adjusted_r2_train = 1 - (1 - r2_train) * (n_train - 1) / (n_train - p - 1)
mse_train = mean_squared_error(y_train, y_train_pred)
rmse_train = np.sqrt(mse_train)

# Test metrics
r2_test = r2_score(y_test, y_test_pred)
n_test = X_test.shape[0]
adjusted_r2_test = 1 - (1 - r2_test) * (n_test - 1) / (n_test - p - 1)
mse_test = mean_squared_error(y_test, y_test_pred)
rmse_test = np.sqrt(mse_test)

print("OLS Regression Results:")
print("\nTraining Set:")
print(f"R²: {r2_train:.4f}")
print(f"Adjusted R²: {adjusted_r2_train:.4f}")
print(f"Mean Squared Error: {mse_train:.4f}")
print(f"Root Mean Squared Error: {rmse_train:.4f}")
print("\nTest Set:")
print(f"R²: {r2_test:.4f}")
print(f"Adjusted R²: {adjusted_r2_test:.4f}")
print(f"Mean Squared Error: {mse_test:.4f}")
print(f"Root Mean Squared Error: {rmse_test:.4f}")

# Step 3: Residual Analysis
# Compute residuals for the full dataset (for spatial analysis)
y_pred_full = ols_model.predict(gdf[features])
residuals = gdf['price_per_sqft'] - y_pred_full
gdf['residuals'] = residuals

# Create spatial weights matrix using k-nearest neighbors (k=5)
w = lp.weights.KNN.from_dataframe(gdf, k=40)

# Compute Moran's I for residuals
moran = Moran(residuals, w)
print("\nResidual Spatial Autocorrelation (Moran's I):")
print(f"Moran's I: {moran.I:.4f}")
print(f"p-value: {moran.p_sim:.4f}")
if moran.p_sim < 0.05:
    print("Significant spatial autocorrelation detected in residuals (p < 0.05).")
else:
    print("No significant spatial autocorrelation in residuals (p >= 0.05).")

# Step 4: Visualizations
# Visualization 1: Residual Scatter Plot (Predicted vs Residuals)
plt.figure(figsize=(8, 6))
sns.scatterplot(x=y_pred_full, y=residuals, alpha=0.5)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel('Predicted Price per Sqft')
plt.ylabel('Residuals')
plt.title('Residuals vs Predicted Values')
plt.savefig('residuals_scatter.png')
plt.close()

# Visualization 2: Spatial Residual Map
fig, ax = plt.subplots(figsize=(10, 8))
gdf.plot(column='residuals', cmap='RdBu', legend=True, ax=ax,
         legend_kwds={'label': "Residuals (Price per Sqft)", 'orientation': "horizontal"})
plt.title('Spatial Distribution of OLS Residuals')
plt.savefig('residuals_map.png')
plt.close()

print("\nVisualizations saved as 'residuals_scatter.png' and 'residuals_map.png'.")

OLS Regression Results:

Training Set:
R²: 0.7092
Adjusted R²: 0.7091
Mean Squared Error: 6640.2117
Root Mean Squared Error: 81.4875

Test Set:
R²: 0.7076
Adjusted R²: 0.7074
Mean Squared Error: 6833.1406
Root Mean Squared Error: 82.6628


/usr/local/lib/python3.11/dist-packages/libpysal/weights/distance.py:153: UserWarning: The weights matrix is not fully connected: 
 There are 72 disconnected components.
  W.__init__(self, neighbors, id_order=ids, **kwargs)



Residual Spatial Autocorrelation (Moran's I):
Moran's I: 0.0011
p-value: 0.1010
No significant spatial autocorrelation in residuals (p >= 0.05).

Visualizations saved as 'residuals_scatter.png' and 'residuals_map.png'.


In [ ]:
from mgwr.gwr import GWR
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Step 1: Check for Duplicate Coordinates
# Check if there are duplicate coordinates in gdf
coords = np.array(list(zip(gdf.geometry.x, gdf.geometry.y)))
unique_coords, counts = np.unique(coords, axis=0, return_counts=True)
if np.any(counts > 1):
    print(f"Warning: {np.sum(counts > 1)} duplicate coordinates found. Removing duplicates.")
    gdf = gdf.drop_duplicates(subset=['x', 'y'], keep='first')
    coords = np.array(list(zip(gdf.geometry.x, gdf.geometry.y)))
else:
    print("No duplicate coordinates found.")

# Step 2: Check Feature Correlations
# Define numerical features for correlation check
numerical_features = ['distance_to_mrt_meters', 'distance_to_cbd', 'distance_to_pri_school_meters',
                      'price_per_sqft_lag', 'floor_area_sqm']
print("\nCorrelation Matrix for Numerical Features:")
print(gdf[numerical_features].corr())

# Step 3: Prepare Data for GWR
# Use reduced feature set to avoid collinearity
features = [
    'distance_to_mrt_meters', 'price_per_sqft_lag','distance_to_cbd', 'distance_to_pri_school_meters',
    'floor_area_sqm'
]

# Standardize all numerical features
scaler = StandardScaler()
gdf[features] = scaler.fit_transform(gdf[features])

X = gdf[features].values
y = gdf['price_per_sqft'].values.reshape((-1, 1))

# Step 4: Set Fixed Bandwidth
# Use fixed bandwidth of 200 neighbors
bw = 200
print(f"Fixed bandwidth (number of nearest neighbors): {bw}")

# Step 5: Fit GWR Model
gwr_model = GWR(coords, y, X, bw, kernel='bisquare')
gwr_results = gwr_model.fit()

# Step 6: Compute GWR Metrics
print("\nGWR Results:")
print(f"Global R²: {gwr_results.R2:.4f}")
print(f"Adjusted R²: {gwr_results.adj_R2:.4f}")
print(f"AIC: {gwr_results.aic:.2f}")
print(f"Effective number of parameters: {gwr_results.ENP:.2f}")

# Step 7: Store Local Results
# Add local R² and coefficients to GeoDataFrame
gdf['local_R2'] = gwr_results.localR2
gdf['local_distance_to_mrt_coeff'] = gwr_results.params[:, features.index('distance_to_mrt_meters')]

# Step 8: Visualizations
# Visualization 1: Local R² Map
fig, ax = plt.subplots(figsize=(10, 8))
gdf.plot(column='local_R2', cmap='YlGn', legend=True, ax=ax,
         legend_kwds={'label': "Local R²", 'orientation': "horizontal"})
plt.title('Spatial Distribution of Local R² (GWR)')
plt.savefig('gwr_local_r2_map.png')
plt.close()

# Visualization 2: Local Coefficient Map for distance_to_mrt_meters
fig, ax = plt.subplots(figsize=(10, 8))
gdf.plot(column='local_distance_to_mrt_coeff', cmap='RdBu', legend=True, ax=ax,
         legend_kwds={'label': "Local Coefficient (Distance to MRT)", 'orientation': "horizontal"})
plt.title('Spatial Variation of Distance to MRT Coefficient (GWR)')
plt.savefig('gwr_mrt_coeff_map.png')
plt.close()

print("\nVisualizations saved as 'gwr_local_r2_map.png' and 'gwr_mrt_coeff_map.png'.")

No duplicate coordinates found.

Correlation Matrix for Numerical Features:
                               distance_to_mrt_meters  distance_to_cbd  \
distance_to_mrt_meters                       1.000000         0.040468   
distance_to_cbd                              0.040468         1.000000   
distance_to_pri_school_meters                0.131119        -0.214178   
price_per_sqft_lag                          -0.169991        -0.601298   
floor_area_sqm                               0.089495         0.190592   

                               distance_to_pri_school_meters  \
distance_to_mrt_meters                              0.131119   
distance_to_cbd                                    -0.214178   
distance_to_pri_school_meters                       1.000000   
price_per_sqft_lag                                  0.206043   
floor_area_sqm                                     -0.087593   

                               price_per_sqft_lag  floor_area_sqm  
distance_to_mrt_meters    

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import hdbscan
from sklearn.preprocessing import StandardScaler

# Step 1: Prepare Data for Clustering
# Select features for clustering: spatial coordinates (x, y) and price_per_sqft
clustering_features = ['x', 'y', 'price_per_sqft']
gdf_clustering = gdf[clustering_features].copy()

# Standardize features to ensure equal weighting
scaler = StandardScaler()
gdf_clustering[clustering_features] = scaler.fit_transform(gdf_clustering[clustering_features])

# Step 2: Apply HDBSCAN Clustering
# Initialize HDBSCAN with min_cluster_size=50 (adjustable based on dataset size)
clusterer = hdbscan.HDBSCAN(min_cluster_size=50, metric='euclidean', cluster_selection_method='eom')
gdf['cluster'] = clusterer.fit_predict(gdf_clustering)

# Count clusters and noise points
n_clusters = len(set(gdf['cluster']) - {-1})  # Exclude noise (-1)
n_noise = sum(gdf['cluster'] == -1)
print(f"HDBSCAN Clustering Results:")
print(f"Number of clusters: {n_clusters}")
print(f"Number of noise points: {n_noise} ({n_noise/len(gdf)*100:.2f}% of total)")

# Step 3: Evaluate Clusters
# A. Planning Area Distribution
# Compute the distribution of planning_area_ura within each cluster
cluster_planning_area = gdf.groupby(['cluster', 'planning_area_ura']).size().unstack(fill_value=0)
print("\nPlanning Area Distribution by Cluster:")
print(cluster_planning_area)

# Normalize to show proportions
cluster_planning_area_prop = cluster_planning_area.div(cluster_planning_area.sum(axis=1), axis=0)
print("\nPlanning Area Proportions by Cluster:")
print(cluster_planning_area_prop)

# B. MRT Proximity Analysis
# Compute mean distance_to_mrt_meters per cluster
mrt_proximity = gdf.groupby('cluster')['distance_to_mrt_meters'].mean().reset_index()
print("\nMean Distance to MRT (meters) by Cluster:")
print(mrt_proximity)

# Step 4: Visualizations
# Visualization 1: Spatial Cluster Map
fig, ax = plt.subplots(figsize=(10, 8))
gdf.plot(column='cluster', cmap='tab20', legend=True, ax=ax,
         legend_kwds={'label': "Cluster ID", 'orientation': "horizontal"})
plt.title('Spatial Distribution of HDBSCAN Clusters')
plt.savefig('hdbscan_cluster_map.png')
plt.close()

# Visualization 2: Planning Area Distribution by Cluster
fig, ax = plt.subplots(figsize=(12, 6))
cluster_planning_area_prop.plot(kind='bar', stacked=True, ax=ax)
plt.title('Planning Area Distribution by Cluster')
plt.xlabel('Cluster ID')
plt.ylabel('Proportion')
plt.legend(title='Planning Area', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig('hdbscan_planning_area_dist.png')
plt.close()

# Visualization 3: MRT Proximity by Cluster
fig, ax = plt.subplots(figsize=(8, 6))
sns.boxplot(x='cluster', y='distance_to_mrt_meters', data=gdf, ax=ax)
plt.title('Distance to MRT by Cluster')
plt.xlabel('Cluster ID')
plt.ylabel('Distance to MRT (meters)')
plt.savefig('hdbscan_mrt_proximity.png')
plt.close()

print("\nVisualizations saved as 'hdbscan_cluster_map.png', 'hdbscan_planning_area_dist.png', and 'hdbscan_mrt_proximity.png'.")

/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


HDBSCAN Clustering Results:
Number of clusters: 3
Number of noise points: 141 (0.20% of total)

Planning Area Distribution by Cluster:
planning_area_ura  ANG MO KIO  BEDOK  BISHAN  BUKIT BATOK  BUKIT MERAH  \
cluster                                                                  
-1                          6      6       5            5            1   
 0                          0      0       0            0            0   
 1                       2419   3041    1427            0            0   
 2                          0      0       0         2953         2966   

planning_area_ura  BUKIT PANJANG  BUKIT TIMAH  CHANGI  CHOA CHU KANG  \
cluster                                                                
-1                             8            0       0              0   
 0                             0            0       0              0   
 1                             0            0       6              0   
 2                          2856          143       0       

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from spreg import ML_Lag
import libpysal as lp

# Step 1: Verify Data Availability
features = ['distance_to_mrt_meters', 'distance_to_cbd', 'distance_to_pri_school_meters',
            'price_per_sqft_lag', 'floor_area_sqm', 'planning_area_ura_encoded', 'flat_type_encoded']
target = 'price_per_sqft'

# Check if required columns exist in gdf
missing_columns = [col for col in features + [target] if col not in gdf.columns]
if missing_columns:
    print(f"Error: Missing columns in gdf: {missing_columns}")
    raise KeyError(f"Missing columns: {missing_columns}")

# Step 2: Random Forests Model
# Initialize and train Random Forest
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# Predict on test set
y_pred_rf = rf_model.predict(X_test)

# Evaluate Random Forest
mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test, y_pred_rf)

print("\nRandom Forest Results:")
print(f"MAE: {mae_rf:.2f}")
print(f"RMSE: {rmse_rf:.2f}")
print(f"R²: {r2_rf:.4f}")

# Feature Importance
feature_importance = pd.DataFrame({
    'Feature': features,
    'Importance': rf_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

print("\nFeature Importance:")
print(feature_importance)

# Step 3: Visualization for Random Forests
# Plot Feature Importance
fig, ax = plt.subplots(figsize=(8, 6))
feature_importance.plot(x='Feature', y='Importance', kind='bar', ax=ax)
plt.title('Random Forest Feature Importance')
plt.xlabel('Feature')
plt.ylabel('Importance')
plt.tight_layout()
plt.savefig('rf_feature_importance.png')
plt.close()

# Plot Predicted vs Actual
fig, ax = plt.subplots(figsize=(8, 6))
plt.scatter(y_test, y_pred_rf, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.title('Random Forest: Predicted vs Actual Price per Sqft')
plt.xlabel('Actual Price per Sqft')
plt.ylabel('Predicted Price per Sqft')
plt.savefig('rf_pred_vs_actual.png')
plt.close()

# Step 4: Spatial Autoregressive (SAR) Model (Optional)
# Create spatial weights matrix (k-nearest neighbors, k=5)
w = lp.weights.KNN.from_dataframe(gdf, k=5)
w.transform = 'r'  # Row-standardize weights

# Prepare data for SAR (use all data, not train-test split, for model estimation)
X_sar = gdf[features].values
y_sar = gdf[target].values

# Fit SAR model (Spatial Lag)
sar_model = ML_Lag(y_sar, X_sar, w, name_y=target, name_x=features)
print("\nSAR Model Summary:")
print(sar_model.summary)

# Predict on test set (using indices from X_test)
test_indices = X_test.index
y_pred_sar = sar_model.predy[test_indices]

# Evaluate SAR
mae_sar = mean_absolute_error(y_test, y_pred_sar)
rmse_sar = np.sqrt(mean_squared_error(y_test, y_pred_sar))
r2_sar = r2_score(y_test, y_pred_sar)

print("\nSAR Model Results (Test Set):")
print(f"MAE: {mae_sar:.2f}")
print(f"RMSE: {rmse_sar:.2f}")
print(f"R²: {r2_sar:.4f}")

# Step 5: Visualization for SAR
# Plot SAR Coefficients
sar_coefs = pd.DataFrame({
    'Feature': ['spatial_lag'] + features,
    'Coefficient': [sar_model.rho] + list(sar_model.betas[:-1].flatten())
})

fig, ax = plt.subplots(figsize=(8, 6))
sar_coefs.plot(x='Feature', y='Coefficient', kind='bar', ax=ax)
plt.title('SAR Model Coefficients')
plt.xlabel('Feature')
plt.ylabel('Coefficient')
plt.tight_layout()
plt.savefig('sar_coefficients.png')
plt.close()

print("\nVisualizations saved as 'rf_feature_importance.png', 'rf_pred_vs_actual.png', and 'sar_coefficients.png'.")


Random Forest Results:
MAE: 39.88
RMSE: 55.53
R²: 0.8681

Feature Importance:
                         Feature  Importance
3             price_per_sqft_lag    0.808335
1                distance_to_cbd    0.047605
0         distance_to_mrt_meters    0.043443
6              flat_type_encoded    0.040920
2  distance_to_pri_school_meters    0.040313
4                 floor_area_sqm    0.014452
5      planning_area_ura_encoded    0.004932


/usr/local/lib/python3.11/dist-packages/libpysal/weights/distance.py:153: UserWarning: The weights matrix is not fully connected: 
 There are 6831 disconnected components.
  W.__init__(self, neighbors, id_order=ids, **kwargs)
